# 2. Cleaning the data

The raw records need four steps before a model can use them
(`fico_xml.preprocessing.clean` applies all of them):

1. **recoding** the two delinquency features to a common scale
   (0 for derogatory comment, 1 for 120+ days, 2 for 90 days, 3 for 60 days,
   4 for 30 days delinquent, 5 for unknown delinquency and 6 for current and
   never delinquent);
2. **special values** are used to set aside the applicants without a credit
   record;
   -9 and -8 become missing values;
   -7 ("condition not met") becomes a missing value too, except for the months
   since the most recent credit check, which is *guessed* from the number of
   credit checks in the last 6 months;
3. **consistency checks** identify records that contradict themselves, which
   are then removed; and
4. **feature engineering** builds new features.

In [1]:
from fico_xml import data, preprocessing

raw = data.load_raw()
clean = preprocessing.clean(raw)
clean.data.shape

(8199, 52)

## Why applicants were removed

Besides the applicants without a credit record, more than a thousand records
contradict themselves;
e.g., more accounts in good standing than accounts in total or a worse payment
delay in the last 12 months than ever:

In [2]:
clean.removed.value_counts()

no_record                                588
more_satisfactory_accounts_than_total    430
delinquency_inconsistent_with_60_days    413
worse_delinquency_recently_than_ever     393
delinquency_inconsistent_with_90_days    123
more_recent_accounts_than_total           96
average_age_above_oldest                  91
more_accounts_with_balance_than_total     90
no_accounts                               29
recent_account_older_than_oldest           7
Name: count, dtype: int64

The rules (and the message an applicant would see):

In [3]:
import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.DataFrame([(r.name, r.message) for r in preprocessing.RULES], columns=['rule', 'message'])

,rule,message
0,recent_account_older_than_oldest,Your most recent credit account is older than your oldest one.
1,average_age_above_oldest,Your credit accounts are on average older than your oldest one.
2,worse_delinquency_recently_than_ever,Your worst payment delay in the last 12 months is worse than your worst payment delay ever.
3,fewer_60_than_90_days_overdue,Fewer of your accounts were ever 60 or more days overdue than 90 or more days overdue.
4,delinquency_inconsistent_with_60_days,"Your worst payment delay ever is shorter than 60 days, but some of your accounts were 60 or more days overdue."
5,delinquency_inconsistent_with_90_days,"Your worst payment delay ever is shorter than 90 days, but some of your accounts were 90 or more days overdue."
6,more_recent_accounts_than_total,More credit accounts were opened in the last 12 months than you have in total.
7,no_accounts,You have no credit accounts.
8,more_accounts_with_balance_than_total,You have more revolving and instalment accounts with a balance than credit accounts in total.
9,more_satisfactory_accounts_than_total,You have more credit accounts in good standing than credit accounts in total.


A record can violate several rules at once:

In [4]:
clean.violations.sum(axis=1).value_counts().sort_index()

0    8199
1     621
2     536
3     380
4     109
5      10
6      11
7       5
Name: count, dtype: int64

Only the consistent applicants are used from now on;
the classes remain balanced:

In [5]:
clean.target.value_counts()

RiskPerformance
Bad     4170
Good    4029
Name: count, dtype: int64

## Missing values after cleaning

Special values turned into missing values;
the features with the most of them:

In [6]:
clean.data.isnull().sum().sort_values(ascending=False).head(8)

MSinceMostRecentDelq                       4013
NetFractionInstallBurden                   2864
MSinceMostRecentInqexcl7days               1807
MSinceMostRecentInqexcl7daysGuess           998
PercInstallTradesWBalanceCalc               699
NumInstallTradesWBalance                    699
NumBank2NatlTradesWHighUtilization          474
PercBank2NatlTradesWHighUtilizationCalc     474
dtype: int64

### The guessed months since the most recent credit check

When an applicant had no credit check in the observed period (-7) but the
record lists checks in the last 6 months, the number of months is guessed
from their number;
the more checks, the more recent the last one is assumed to be
(1 check -> 6 months, ..., 6 checks -> 1 month, more -> 0):

In [7]:
pd.crosstab(raw.loc[clean.data.index, 'MSinceMostRecentInqexcl7days'] == -7,
            clean.data['MSinceMostRecentInqexcl7daysGuess'].isnull(),
            rownames=['no credit check (-7)'], colnames=['guess missing'])

guess missing,False,True
no credit check (-7),,
False,6393,386
True,808,612


## Engineered features

Percentages become fractions;
features relative to the total number of accounts, Boolean indicators and
one-hot encoded delinquency categories are added:

In [8]:
engineered = [c for c in clean.data.columns if c.endswith('Calc')]
clean.data[engineered].describe().T[['mean', 'min', 'max']].round(2)

,mean,min,max
PercRevolvingTradesWBalanceCalc,0.19,0.00,1.00
PercInstallTradesWBalanceCalc,0.12,0.01,1.00
PercSatisfactoryTradesCalc,0.89,0.00,1.00
PercTrades60Ever2DerogPubRecCalc,0.03,0.00,1.44
PercTrades90Ever2DerogPubRecCalc,0.02,0.00,1.00
PercTradesOpeninLast12MCalc,0.09,0.00,1.00
PercBank2NatlTradesWHighUtilizationCalc,0.05,0.00,0.75
NumInqLast7DCalc,0.06,0.00,3.00
BoolNverDeliquentCalc,0.47,0.00,1.00
BoolAlwaysSatisfactoryCalc,0.31,0.00,1.00


## The original cleaning

`clean(raw, legacy=True)` reproduces the cleaning of the original analysis.
On these data both keep the same applicants with the same values;
the original table only has one more column, `PercentTradesNeverDelqCalc`
(see the changelog):

In [9]:
legacy = preprocessing.clean(raw, legacy=True)
common = [c for c in clean.data.columns if c in legacy.data.columns]
same_applicants = legacy.data.index.equals(clean.data.index)
changed = (legacy.data[common].fillna(-1) != clean.data[common].fillna(-1)).sum()
same_applicants, changed[changed > 0]

(True, Series([], dtype: int64))